# Lesson 2: 🎨 Maps and timelines 

### 🎯 Learning Objectives
In this lesson, we will use the filtered Argo index file (created in Lesson 1) to visualize and understand the trajectories of the floats on a map and their timelines. Doing so will help us identify the floats and profiles we want.

### 🛠️ Prerequisites
Before starting this lesson, make sure that you have completed **Lesson 1**.

### ❓ How to Use This Notebook
* 📚 **Read** the tutorial text blocks carefully, as they provide the essential background information behind the code.
* ▶️ **Run** each code cell sequentially by clicking the cell and pressing `Shift + Enter`.
* 📝 **Exercise** your knowledge! At the end of this notebook, we provide active learning exercises where you will need to write the code yourself.

### 🚀 Ready? Let's Get Started!
---

## 📚 Tutorials

### Import libraries
▶️ Run the cell below to import relevant Python libraries. **NOTE** we import the functions created in Lesson 1 because we want to use them in this lesson.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
import plotly.express as px

### Load the filtered index

▶️ Run the cell below to load the filtered index file created in Lesson 1.

In [ ]:
df_index = pd.read_parquet('../index_files/argo_synthetic-profile_index_default.parquet')
df_index

### The function `show_map_timeline`

▶️ Run the cell below to define a function that draws an interactive map showing the floats' trajectories and the corresponding timeline. This function utilizes [Plotly](https://plotly.com/python/), which is an open-source graphing library.

In [ ]:
def show_map_timeline(df_index, title_suffix=''):
    """
    Creates and displays an interactive 3D globe map and an interactive timeline
    directly inside the Jupyter Notebook.
    
    INPUT:
    * df_index: the filtered index file (pandas dataframe)
    * title_suffix (optional): string to append to the plot titles
    """
    
    # Pre-sort the entire DataFrame once
    df_index = df_index.sort_values(['wmo', 'datetime'])
    
    # --- 1. Figure Setup ---
    fig_map = go.Figure()
    fig_timeline = go.Figure()
    
    # We use Plotly's built-in categorical color palette (similar to Matplotlib's tab10)
    colors = px.colors.qualitative.Plotly 
    
    # --- 2. The Plotting Loop ---
    for i, (wmo, group) in enumerate(df_index.groupby('wmo')):
        
        color = colors[i % len(colors)]
        label = f"{wmo}"
        
        # ------------------------------------
        # MAP: Plot Trajectory
        # ------------------------------------
        fig_map.add_trace(go.Scattergeo(
            lon=group['longitude'],
            lat=group['latitude'],
            mode='markers',
            marker=dict(size=4, color=color),
            name=label,
            # 1. Hand the datetime column to Plotly so it holds it in memory
            customdata=group['datetime'],   
            # 2. Use %{customdata} to display it, and format it nicely as a date!
            hovertemplate="<b>WMO:</b> " + str(wmo) + "<br><b>Date:</b> %{customdata|%Y-%m-%d %H:%M}<extra></extra>"
        ))
        
        # MAP: Plot Last Coordinate Label (The black number)
        last_lon = group['longitude'].iloc[-1]
        last_lat = group['latitude'].iloc[-1]
        
        fig_map.add_trace(go.Scattergeo(
            lon=[last_lon],
            lat=[last_lat],
            mode='markers+text',
            marker=dict(size=8, color='black'),
            showlegend=False,
            hoverinfo='skip'
        ))
        
        # ------------------------------------
        # TIMELINE: Plot Time Series
        # ------------------------------------
        fig_timeline.add_trace(go.Scatter(
            x=group['datetime'],
            y=[label] * len(group), # Categorical y-axis (WMO strings)
            mode='markers',
            # 'line-ns' creates a vertical line | symbol, matching your old plot
            marker=dict(symbol='line-ns', size=14, color=color, line=dict(width=1.5, color=color)),
            name=label,
            # Formats the popup box to show the exact date cleanly
            hovertemplate="<b>Date:</b> %{x|%Y-%m-%d %H:%M}<extra></extra>"
        ))
        
    # --- 3. Final Formatting ---
    
    # Format the Plotly Map (3D Globe)
    fig_map.update_geos(
        projection_type="orthographic",
        showocean=True, oceancolor="LightBlue",
        showland=True, landcolor="LightGray",
        showcoastlines=True, coastlinecolor="DarkGray",
        # Explicitly turn on the grid for both axes:
        lonaxis_showgrid=True, lonaxis_gridcolor="White",
        lataxis_showgrid=True, lataxis_gridcolor="White"
    )
    
    # Rotate the globe so the data is front and center
    fig_map.update_layout(
        title=f"BGC-Argo Float Trajectories {title_suffix}".strip(),
        geo=dict(
            projection=dict(
                rotation=dict(
                    lon=df_index['longitude'].mean(), 
                    lat=df_index['latitude'].mean()
                )
            )
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        height=600 # Explicit height so it fills the notebook cell nicely
    )
    
    # Format the Plotly Timeline
    fig_timeline.update_layout(
        title=f"BGC-Argo Float Timeline {title_suffix}".strip(),
        xaxis_title="Date",
        yaxis_title="WMO",
        margin=dict(l=0, r=0, b=0, t=40),
        height=400,
        showlegend=False
    )
    
    # --- 4. Display the plots ---
    fig_map.show()
    fig_timeline.show()

### Draw the map and timeline 

▶️ **Run** the cell below to use the function `show_map_timeline`.

In [ ]:
show_map_timeline(df_index)

☝️ Do you see a nice global map with three floats and a timeline? Here are a few cool features about the map and timeline:

* The upper right panel shows a few things you can do.
* You can zoom in/out and adjust the extent of the display.
* You can save it as a png file.
* The black dot shows the last (latest) profiling location of the float.
* Locate the cursor on the colored dot (map) or the vertical line (timeline) to see the date and time of the sampling for the profile.

**This is the end of the tutorials for this lesson. Hope you enjoyed it!**


---

## 📝 Exercises

Using the filtered index files created in Lesson 1 with the three different use cases, create the corresponding maps and timelines using the function `show_map_timeline` (**TIP**: Provide the optional argument `note` so that the function will not overwrite the figures created in the tutorials above).

### Exercise 1: Chlorophyll-a measurements in the global ocean on March 1, 2026

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

df_index = pd.read_parquet('../index_files/argo_synthetic-profile_index_ex1.parquet')
note = 'ex1'
show_map_timeline(df_index, note)

### Exercise 2: *Full-sensor* BGC-Argo floats in the Indian Ocean at least once per month and at least for a year between 2018 and 2025

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

df_index = pd.read_parquet('../index_files/argo_synthetic-profile_index_ex2.parquet')
note = 'ex2'
show_map_timeline(df_index, note)

### Exercise 3: Floats that measured both dissolved oxygen and nitrate and drifted at a speed of nore more than 0.05 m/s in the North Pacific between 2019 and 2022

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

df_index = pd.read_parquet('../index_files/argo_synthetic-profile_index_ex3.parquet')
note = 'ex3'
show_map_timeline(df_index, note)


---

This is the end of the lesson. If you are using **Binder**, don't forget to dowload the files created in this lesson before you lose connection!

Well done 🎉, take a break 💤, have another cup ☕, and move to the next lesson ✍️ when you are ready 💪

While your memory is fresh, please feel free to provide your user experience on this lesson by visiting [this link](https://forms.gle/oAGmz5RTW4Pp46bt7). Thanks!